In [ ]:
import re
from collections import Counter
from itertools import combinations

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False
})

In [ ]:
from google.colab import files

uploaded = files.upload()
csv_filename = "FINAL_US_BA_3606_JOBS.csv"

if csv_filename not in uploaded:
    raise FileNotFoundError(
        f"Please upload {csv_filename}. You selected: {list(uploaded.keys())}"
    )

raw = pd.read_csv(csv_filename)
print("Loaded:", csv_filename)
print("Rows:", len(raw))
print("Columns:", list(raw.columns))

In [ ]:
# scope validation

required_columns = {"job_link", "job_location", "job_skills"}
missing = required_columns - set(raw.columns)
if missing:
    raise ValueError(f"Dataset is missing required columns: {missing}")

raw["job_link"] = raw["job_link"].astype(str)
raw["job_location"] = raw["job_location"].fillna("").astype(str)

country_subdomain = raw["job_link"].str.contains(
    r"^https?://mx\.linkedin\.com", case=False, regex=True
)
mexico_location = raw["job_location"].str.contains(
    r"\bMexico\b", case=False, regex=True
)

out_of_scope = country_subdomain | mexico_location
jobs = raw.loc[~out_of_scope].copy().drop_duplicates("job_link").reset_index(drop=True)

print(f"Rows in uploaded file: {len(raw):,}")
print(f"Records excluded by US-scope check: {int(out_of_scope.sum()):,}")
print(f"Final analytical advertisements: {jobs.job_link.nunique():,}")

if jobs.job_link.nunique() != 3606:
    print("WARNING: The resulting sample is not 3,606. Check the source file/version before using the results.")

In [ ]:
# formatting normalisation
def clean_format(label):
    if pd.isna(label):
        return ""
    value = str(label).strip().lower()
    value = value.replace("’", "'").replace("‘", "'")
    value = re.sub(r"^[*\-•\s]+", "", value)
    value = re.sub(r"[.:;]+$", "", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value

# only clear naming variants are merged.
SYNONYMS = {'communication skills': 'communication', 'problem solving skills': 'problem solving', 'problemsolving skills': 'problem solving', 'teamwork skills': 'teamwork', 'collaboration skills': 'collaboration', 'excel skills': 'excel', 'documentation skills': 'documentation', 'negotiation skills': 'negotiation', 'leadership skills': 'leadership', 'multitasking skills': 'multitasking', 'time management skills': 'time management', 'critical thinking skills': 'critical thinking', 'it': 'information technology', 'ms project': 'microsoft project', 'microsoft excel': 'excel', 'ms excel': 'excel', 'microsoft powerpoint': 'powerpoint', 'ms powerpoint': 'powerpoint', 'microsoft word': 'word', 'ms word': 'word', 'microsoft visio': 'visio', 'ms visio': 'visio', 'microsoft outlook': 'outlook', 'ms outlook': 'outlook', 'ms office': 'microsoft office suite', 'ms office suite': 'microsoft office suite', 'microsoft office': 'microsoft office suite', 'microsoft power bi': 'power bi', 'ms power bi': 'power bi', 'microsoft sql server': 'sql server', 'ms sql server': 'sql server', 'atlassian jira': 'jira', 'jira software': 'jira', 'structured query language (sql)': 'sql', 'agile methodology': 'agile', 'agile methodologies': 'agile', 'uat': 'user acceptance testing', 'user acceptance testing (uat)': 'user acceptance testing', 'uat (user acceptance testing)': 'user acceptance testing', 'uat testing': 'user acceptance testing', 'sdlc': 'software development lifecycle', 'software development life cycle (sdlc)': 'software development lifecycle', 'software development lifecycle (sdlc)': 'software development lifecycle', 'user story': 'user stories', 'requirement gathering': 'requirements gathering', 'data visualisation': 'data visualization', 'data visualisations': 'data visualization', 'data visualizations': 'data visualization'}

def spacing_key(label):
    key = re.sub(r"(?<=\w)[\s\-'.](?=\w)", "", label)
    key = re.sub(r"[\s\-']", "", key)
    if len(key) >= 6 and key.endswith("s") and not key.endswith("ss"):
        key = key[:-1]
    return key

# parse the comma-separated skills field into one row per ad-skill mention
skill_rows = []
for link, skill_text in zip(jobs["job_link"], jobs["job_skills"].fillna("")):
    for original in str(skill_text).split(","):
        formatted = clean_format(original)
        if formatted:
            synonym = SYNONYMS.get(formatted, formatted)
            skill_rows.append((link, original.strip(), formatted, synonym))

mentions = pd.DataFrame(
    skill_rows,
    columns=["job_link", "skill_original", "skill_clean", "skill_after_synonyms"]
)

# group spelling/spacing variants
frequencies = mentions["skill_after_synonyms"].value_counts().to_dict()
groups = {}
for label in frequencies:
    groups.setdefault(spacing_key(label), []).append(label)

canonical = {}
for labels in groups.values():
    chosen = sorted(labels, key=lambda x: (-frequencies[x], len(x), x))[0]
    for label in labels:
        canonical[label] = chosen

mentions["skill_standard"] = mentions["skill_after_synonyms"].map(canonical)

# each ad counts once for each standardised skill
job_skill = mentions[["job_link", "skill_standard"]].drop_duplicates().reset_index(drop=True)

print("Raw skill mentions:", len(mentions))
print("Unique advertisement-skill pairs:", len(job_skill))
print("Distinct standardised skill labels:", job_skill.skill_standard.nunique())
display(mentions.head())

In [ ]:
# calculate job-level skill prevalence
N = jobs["job_link"].nunique()
skill_counts = (
    job_skill.groupby("skill_standard")["job_link"]
    .nunique()
    .rename("n_ads")
    .reset_index()
)
skill_counts["prevalence_pct"] = (100 * skill_counts["n_ads"] / N).round(2)
skill_counts = skill_counts.sort_values(
    ["n_ads", "skill_standard"], ascending=[False, True]
).reset_index(drop=True)
skill_counts.insert(0, "rank", np.arange(1, len(skill_counts) + 1))

top20 = skill_counts.head(20).copy()
display(top20)

plt.figure(figsize=(10, 7))
plot_data = top20.sort_values("prevalence_pct")
plt.barh(plot_data["skill_standard"], plot_data["prevalence_pct"])
plt.xlabel("Share of advertisements (%)")
plt.title("Top 20 standardised skills in US Business Analyst advertisements")
plt.tight_layout()
plt.show()

In [ ]:
CATEGORY_MAP = {'business analysis': 'BA practice (requirements, analysis, process)', 'business analyst': 'BA practice (requirements, analysis, process)', 'it business analyst': 'BA practice (requirements, analysis, process)', 'requirements gathering': 'BA practice (requirements, analysis, process)', 'business requirements': 'BA practice (requirements, analysis, process)', 'business requirements gathering': 'BA practice (requirements, analysis, process)', 'business requirements analysis': 'BA practice (requirements, analysis, process)', 'business requirements collection': 'BA practice (requirements, analysis, process)', 'functional requirements': 'BA practice (requirements, analysis, process)', 'requirements analysis': 'BA practice (requirements, analysis, process)', 'requirements elicitation': 'BA practice (requirements, analysis, process)', 'requirements documentation': 'BA practice (requirements, analysis, process)', 'requirements management': 'BA practice (requirements, analysis, process)', 'technical requirements': 'BA practice (requirements, analysis, process)', 'nonfunctional requirements': 'BA practice (requirements, analysis, process)', 'acceptance criteria': 'BA practice (requirements, analysis, process)', 'functional specifications': 'BA practice (requirements, analysis, process)', 'functional design documents': 'BA practice (requirements, analysis, process)', 'functional design': 'BA practice (requirements, analysis, process)', 'technical specifications': 'BA practice (requirements, analysis, process)', 'user stories': 'BA practice (requirements, analysis, process)', 'use cases': 'BA practice (requirements, analysis, process)', 'stakeholder management': 'BA practice (requirements, analysis, process)', 'stakeholder engagement': 'BA practice (requirements, analysis, process)', 'stakeholder communication': 'BA practice (requirements, analysis, process)', 'gap analysis': 'BA practice (requirements, analysis, process)', 'fitgap analysis': 'BA practice (requirements, analysis, process)', 'feasibility analysis': 'BA practice (requirements, analysis, process)', 'scope definition': 'BA practice (requirements, analysis, process)', 'as is analysis': 'BA practice (requirements, analysis, process)', 'business process analysis': 'BA practice (requirements, analysis, process)', 'business process improvement': 'BA practice (requirements, analysis, process)', 'business process mapping': 'BA practice (requirements, analysis, process)', 'business process modeling': 'BA practice (requirements, analysis, process)', 'business process reengineering': 'BA practice (requirements, analysis, process)', 'business process documentation': 'BA practice (requirements, analysis, process)', 'business processes': 'BA practice (requirements, analysis, process)', 'process improvement': 'BA practice (requirements, analysis, process)', 'process mapping': 'BA practice (requirements, analysis, process)', 'process flows': 'BA practice (requirements, analysis, process)', 'process analysis': 'BA practice (requirements, analysis, process)', 'process documentation': 'BA practice (requirements, analysis, process)', 'process optimization': 'BA practice (requirements, analysis, process)', 'continuous improvement': 'BA practice (requirements, analysis, process)', 'workflows': 'BA practice (requirements, analysis, process)', 'documentation': 'BA practice (requirements, analysis, process)', 'technical documentation': 'BA practice (requirements, analysis, process)', 'user manuals': 'BA practice (requirements, analysis, process)', 'technical writing': 'BA practice (requirements, analysis, process)', 'prototyping': 'BA practice (requirements, analysis, process)', 'wireframes': 'BA practice (requirements, analysis, process)', 'solution design': 'BA practice (requirements, analysis, process)', 'solution proposal': 'BA practice (requirements, analysis, process)', 'solution development': 'BA practice (requirements, analysis, process)', 'system analysis': 'BA practice (requirements, analysis, process)', 'systems analysis': 'BA practice (requirements, analysis, process)', 'system design': 'BA practice (requirements, analysis, process)', 'root cause analysis': 'BA practice (requirements, analysis, process)', 'costbenefit analysis': 'BA practice (requirements, analysis, process)', 'subject matter experts': 'BA practice (requirements, analysis, process)', 'liaison between end users and development teams': 'BA practice (requirements, analysis, process)', 'liaison': 'BA practice (requirements, analysis, process)', 'business/system/user issues resolution': 'BA practice (requirements, analysis, process)', 'sql': 'Data & technical', 'sql queries': 'Data & technical', 'sql server': 'Data & technical', 'advanced queries': 'Data & technical', 'data analysis': 'Data & technical', 'data analytics': 'Data & technical', 'analytics': 'Data & technical', 'business analytics': 'Data & technical', 'business intelligence': 'Data & technical', 'excel': 'Data & technical', 'vlookups': 'Data & technical', 'pivot tables': 'Data & technical', 'power bi': 'Data & technical', 'tableau': 'Data & technical', 'data visualization': 'Data & technical', 'dashboards': 'Data & technical', 'reporting': 'Data & technical', 'reports': 'Data & technical', 'report writing': 'Data & technical', 'python': 'Data & technical', 'r': 'Data & technical', 'sas': 'Data & technical', 'java': 'Data & technical', 'javascript': 'Data & technical', 'aws': 'Data & technical', 'azure': 'Data & technical', 'cloud computing': 'Data & technical', 'machine learning': 'Data & technical', 'data science': 'Data & technical', 'statistics': 'Data & technical', 'statistical analysis': 'Data & technical', 'quantitative analysis': 'Data & technical', 'data mining': 'Data & technical', 'data modeling': 'Data & technical', 'data management': 'Data & technical', 'data mapping': 'Data & technical', 'data integration': 'Data & technical', 'data migration': 'Data & technical', 'data conversion': 'Data & technical', 'data extraction': 'Data & technical', 'data extracts': 'Data & technical', 'data loading': 'Data & technical', 'data warehouse': 'Data & technical', 'data warehousing': 'Data & technical', 'data governance': 'Data & technical', 'data quality': 'Data & technical', 'data integrity': 'Data & technical', 'data validation': 'Data & technical', 'data collection': 'Data & technical', 'data interpretation': 'Data & technical', 'analysis of data': 'Data & technical', 'information extraction': 'Data & technical', 'information processing': 'Data & technical', 'database management systems': 'Data & technical', 'database management systems knowledge': 'Data & technical', 'database management': 'Data & technical', 'relational databases': 'Data & technical', 'databases': 'Data & technical', 'database user interface': 'Data & technical', 'database user interface knowledge': 'Data & technical', 'microsoft office suite': 'Data & technical', 'powerpoint': 'Data & technical', 'word': 'Data & technical', 'outlook': 'Data & technical', 'visio': 'Data & technical', 'access': 'Data & technical', 'sharepoint': 'Data & technical', 'servicenow': 'Data & technical', 'software development': 'Data & technical', 'development': 'Data & technical', 'application development': 'Data & technical', 'devops': 'Data & technical', 'azure devops': 'Data & technical', 'automation': 'Data & technical', 'configuration': 'Data & technical', 'system configuration': 'Data & technical', 'system integration': 'Data & technical', 'integration': 'Data & technical', 'information technology': 'Data & technical', 'information systems': 'Data & technical', 'web services': 'Data & technical', 'cloud': 'Data & technical', 'edi': 'Data & technical', 'project management': 'Delivery & project', 'project planning': 'Delivery & project', 'project timelines': 'Delivery & project', 'program management': 'Delivery & project', 'agile': 'Delivery & project', 'agile development': 'Delivery & project', 'scrum': 'Delivery & project', 'scrum master': 'Delivery & project', 'kanban': 'Delivery & project', 'waterfall': 'Delivery & project', 'jira': 'Delivery & project', 'confluence': 'Delivery & project', 'testing': 'Delivery & project', 'user acceptance testing': 'Delivery & project', 'unit testing': 'Delivery & project', 'integration testing': 'Delivery & project', 'regression testing': 'Delivery & project', 'system testing': 'Delivery & project', 'software testing': 'Delivery & project', 'functional testing': 'Delivery & project', 'test plans': 'Delivery & project', 'test cases': 'Delivery & project', 'test planning': 'Delivery & project', 'test data preparation': 'Delivery & project', 'data conversion & integration testing': 'Delivery & project', 'icrp testing': 'Delivery & project', 'crp testing': 'Delivery & project', 'quality assurance': 'Delivery & project', 'change management': 'Delivery & project', 'risk management': 'Delivery & project', 'risk assessment': 'Delivery & project', 'software development lifecycle': 'Delivery & project', 'implementation': 'Delivery & project', 'system implementation': 'Delivery & project', 'migration projects': 'Delivery & project', 'upgrade projects': 'Delivery & project', 'integration projects': 'Delivery & project', 'deployment': 'Delivery & project', 'migration': 'Delivery & project', 'product management': 'Delivery & project', 'vendor management': 'Delivery & project', 'user training': 'Delivery & project', 'functional training': 'Delivery & project', 'enduser training': 'Delivery & project', 'training': 'Delivery & project', 'support': 'Delivery & project', 'project status communication': 'Delivery & project', 'microsoft project': 'Delivery & project', 'unit and integration testing': 'Delivery & project', 'communication': 'Interpersonal & communication', 'written communication': 'Interpersonal & communication', 'verbal communication': 'Interpersonal & communication', 'oral communication': 'Interpersonal & communication', 'teamwork': 'Interpersonal & communication', 'team collaboration': 'Interpersonal & communication', 'collaboration': 'Interpersonal & communication', 'crossfunctional collaboration': 'Interpersonal & communication', 'crossfunctional department collaboration': 'Interpersonal & communication', 'crossfunctional teams': 'Interpersonal & communication', 'team player': 'Interpersonal & communication', 'team leadership': 'Interpersonal & communication', 'leadership': 'Interpersonal & communication', 'mentoring': 'Interpersonal & communication', 'interpersonal skills': 'Interpersonal & communication', 'presentation skills': 'Interpersonal & communication', 'presentation': 'Interpersonal & communication', 'negotiation': 'Interpersonal & communication', 'facilitation': 'Interpersonal & communication', 'relationship building': 'Interpersonal & communication', 'customer service': 'Interpersonal & communication', 'consulting': 'Interpersonal & communication', 'writing': 'Interpersonal & communication', 'problem solving': 'Analytical thinking & work effectiveness', 'complex problemsolving skills': 'Analytical thinking & work effectiveness', 'complex problemsolving': 'Analytical thinking & work effectiveness', 'problem resolution': 'Analytical thinking & work effectiveness', 'analytical skills': 'Analytical thinking & work effectiveness', 'analytical thinking': 'Analytical thinking & work effectiveness', 'critical thinking': 'Analytical thinking & work effectiveness', 'attention to detail': 'Analytical thinking & work effectiveness', 'detailoriented': 'Analytical thinking & work effectiveness', 'decision making': 'Analytical thinking & work effectiveness', 'time management': 'Analytical thinking & work effectiveness', 'multitasking': 'Analytical thinking & work effectiveness', 'adaptability': 'Analytical thinking & work effectiveness', 'flexibility': 'Analytical thinking & work effectiveness', 'flexible': 'Analytical thinking & work effectiveness', 'prioritization': 'Analytical thinking & work effectiveness', 'workload prioritization': 'Analytical thinking & work effectiveness', 'organizational skills': 'Analytical thinking & work effectiveness', 'organization': 'Analytical thinking & work effectiveness', 'independent work': 'Analytical thinking & work effectiveness', 'ability to work independently': 'Analytical thinking & work effectiveness', 'selfmotivation': 'Analytical thinking & work effectiveness', 'initiative': 'Analytical thinking & work effectiveness', 'creativity': 'Analytical thinking & work effectiveness', 'innovation': 'Analytical thinking & work effectiveness', 'strategic thinking': 'Analytical thinking & work effectiveness', 'strategic planning': 'Analytical thinking & work effectiveness', 'research': 'Analytical thinking & work effectiveness', 'market research': 'Analytical thinking & work effectiveness', 'fastpaced environment': 'Analytical thinking & work effectiveness', 'integrity': 'Analytical thinking & work effectiveness', 'troubleshooting': 'Analytical thinking & work effectiveness', 'analysis': 'Analytical thinking & work effectiveness', 'planning': 'Analytical thinking & work effectiveness', 'technical skills': 'Analytical thinking & work effectiveness', 'design skills': 'Analytical thinking & work effectiveness', 'design': 'Analytical thinking & work effectiveness', 'business acumen': 'Analytical thinking & work effectiveness', 'finance': 'Business domain & enterprise systems', 'forecasting': 'Business domain & enterprise systems', 'accounting': 'Business domain & enterprise systems', 'financial analysis': 'Business domain & enterprise systems', 'financial reporting': 'Business domain & enterprise systems', 'financial modeling': 'Business domain & enterprise systems', 'financial services': 'Business domain & enterprise systems', 'budgeting': 'Business domain & enterprise systems', 'procurement': 'Business domain & enterprise systems', 'inventory management': 'Business domain & enterprise systems', 'demand planning': 'Business domain & enterprise systems', 'manufacturing': 'Business domain & enterprise systems', 'supply chain management': 'Business domain & enterprise systems', 'distribution': 'Business domain & enterprise systems', 'product data management': 'Business domain & enterprise systems', 'shop floor management': 'Business domain & enterprise systems', 'shop floor': 'Business domain & enterprise systems', 'material & capacity planning': 'Business domain & enterprise systems', 'mrp & mps': 'Business domain & enterprise systems', 'product costing & manufacturing accounting': 'Business domain & enterprise systems', 'discrete & lean manufacturing': 'Business domain & enterprise systems', 'lean manufacturing': 'Business domain & enterprise systems', 'quality management': 'Business domain & enterprise systems', 'healthcare': 'Business domain & enterprise systems', 'sales': 'Business domain & enterprise systems', 'marketing': 'Business domain & enterprise systems', 'compliance': 'Business domain & enterprise systems', 'regulatory compliance': 'Business domain & enterprise systems', 'six sigma': 'Business domain & enterprise systems', 'engineering': 'Business domain & enterprise systems', 'erp': 'Business domain & enterprise systems', 'erp systems': 'Business domain & enterprise systems', 'erp implementation': 'Business domain & enterprise systems', 'erp systems knowledge': 'Business domain & enterprise systems', 'sap': 'Business domain & enterprise systems', 'oracle': 'Business domain & enterprise systems', 'salesforce': 'Business domain & enterprise systems', 'workday': 'Business domain & enterprise systems', 'crm': 'Business domain & enterprise systems', 'bip output': 'Business domain & enterprise systems', 'e1 interactive applications': 'Business domain & enterprise systems', 'e1 batch application': 'Business domain & enterprise systems', 'uxone udo': 'Business domain & enterprise systems', 'jde e1': 'Business domain & enterprise systems', 'orchestrator solutions': 'Business domain & enterprise systems', 'watchlist': 'Business domain & enterprise systems', 'e1 pages': 'Business domain & enterprise systems', 'jde sales': 'Business domain & enterprise systems', 'jde certified in manufacturing': 'Business domain & enterprise systems', 'menus': 'Business domain & enterprise systems', 'jde oracle support': 'Business domain & enterprise systems', 'jde edwards': 'Business domain & enterprise systems', 'jde': 'Business domain & enterprise systems', 'jde world': 'Business domain & enterprise systems', 'processing options': 'Business domain & enterprise systems', 'version creation': 'Business domain & enterprise systems', 'jde mapping': 'Business domain & enterprise systems', 'integration with other modules': 'Business domain & enterprise systems', 'jde e1 inventory': 'Business domain & enterprise systems', 'production promotion': 'Business domain & enterprise systems', 'mapping jde with existing system': 'Business domain & enterprise systems', 'jde e1 8.10 and above': 'Business domain & enterprise systems', 'retrofitting of modifications': 'Business domain & enterprise systems', 'menu creation': 'Business domain & enterprise systems', 'creating menus & versions': 'Business domain & enterprise systems', 'versions': 'Business domain & enterprise systems', 'module integration': 'Business domain & enterprise systems', 'jde modules': 'Business domain & enterprise systems', 'conference room pilot': 'Business domain & enterprise systems', 'setups': 'Business domain & enterprise systems', 'business': 'Business domain & enterprise systems', 'system health check': 'Business domain & enterprise systems', "bachelor's degree": 'Education & credentials', "master's degree": 'Education & credentials', "bachelor's or master's degree": 'Education & credentials', 'computer science': 'Education & credentials', 'business administration': 'Education & credentials', 'economics': 'Education & credentials', 'cbap': 'Education & credentials', 'project management professional (pmp)': 'Education & credentials', 'pmp': 'Education & credentials', 'business analysis certification': 'Education & credentials'}

In [ ]:
# apply the primary category mapping to the reviewed skills
MIN_ADS_REVIEWED = 36
EDUCATION_CATEGORY = "Education & credentials"
UNCLASSIFIED = "Unclassified"

skill_counts["category"] = skill_counts["skill_standard"].map(CATEGORY_MAP).fillna(UNCLASSIFIED)
reviewed = skill_counts[skill_counts["n_ads"] >= MIN_ADS_REVIEWED].copy()
reviewed["category"] = reviewed["skill_standard"].map(CATEGORY_MAP).fillna(UNCLASSIFIED)

capability_categories = [
    "BA practice (requirements, analysis, process)",
    "Data & technical",
    "Delivery & project",
    "Interpersonal & communication",
    "Analytical thinking & work effectiveness",
    "Business domain & enterprise systems"
]

# build ad-by-category flags
job_skill_category = job_skill.merge(
    reviewed[["skill_standard", "category"]], on="skill_standard", how="left"
)
job_skill_category["category"] = job_skill_category["category"].fillna(UNCLASSIFIED)
ad_category = (
    job_skill_category[job_skill_category["category"].isin(capability_categories)]
    .drop_duplicates(["job_link", "category"])
    .assign(present=1)
    .pivot(index="job_link", columns="category", values="present")
    .reindex(index=jobs["job_link"], columns=capability_categories)
    .fillna(0)
    .astype(int)
)

category_summary = pd.DataFrame({
    "category": capability_categories,
    "ads_with_category": [int(ad_category[c].sum()) for c in capability_categories],
    "prevalence_pct": [round(100 * ad_category[c].mean(), 2) for c in capability_categories],
    "skills_mapped": [
        int((reviewed["category"] == c).sum()) for c in capability_categories
    ]
}).sort_values("prevalence_pct", ascending=False)

display(category_summary)

plt.figure(figsize=(10, 6))
plot_data = category_summary.sort_values("prevalence_pct")
plt.barh(plot_data["category"], plot_data["prevalence_pct"])
plt.xlabel("Share of advertisements (%)")
plt.title("Prevalence of capability categories")
plt.tight_layout()
plt.show()

# number of capability categories represented in each advertisement
category_breadth = ad_category.sum(axis=1)
breadth_counts = (
    category_breadth.value_counts()
    .reindex(range(0, len(capability_categories) + 1), fill_value=0)
    .sort_index()
)
display(pd.DataFrame({
    "number_of_capability_categories": breadth_counts.index,
    "n_ads": breadth_counts.values,
    "pct_of_ads": (100 * breadth_counts.values / N).round(2)
}))

plt.figure(figsize=(8, 5))
plt.bar(breadth_counts.index.astype(str), breadth_counts.values)
plt.xlabel("Number of capability categories in an ad")
plt.ylabel("Number of advertisements")
plt.title("Breadth of capability categories per advertisement")
plt.tight_layout()
plt.show()

In [ ]:
# co-occurrence among skills appearing in at least 5% of ads
threshold = int(np.ceil(0.05 * N))
frequent = skill_counts.loc[skill_counts["n_ads"] >= threshold, "skill_standard"].tolist()

binary = (
    job_skill[job_skill["skill_standard"].isin(frequent)]
    .assign(value=1)
    .pivot(index="job_link", columns="skill_standard", values="value")
    .reindex(index=jobs["job_link"])
    .fillna(0)
    .astype(int)
)

pair_rows = []
for skill_a, skill_b in combinations(binary.columns, 2):
    both = int(((binary[skill_a] == 1) & (binary[skill_b] == 1)).sum())
    n_a = int(binary[skill_a].sum())
    n_b = int(binary[skill_b].sum())
    either = n_a + n_b - both
    pair_rows.append({
        "skill_A": skill_a,
        "skill_B": skill_b,
        "n_ads_both": both,
        "pct_of_ads_both": round(100 * both / N, 2),
        "jaccard": round(both / either, 3) if either else 0,
        "lift": round((both / N) / ((n_a / N) * (n_b / N)), 2) if n_a and n_b else 0
    })

pairs = pd.DataFrame(pair_rows).sort_values(
    ["n_ads_both", "jaccard"], ascending=False
).reset_index(drop=True)
display(pairs.head(20))

In [ ]:
# co-occurrence heatmap for the 20 most prevalent skills in the >=5% group
heatmap_skills = skill_counts.loc[
    skill_counts["skill_standard"].isin(frequent), "skill_standard"
].head(20).tolist()
co_matrix = binary[heatmap_skills].T.dot(binary[heatmap_skills])

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(co_matrix.values, aspect="auto")
ax.set_xticks(range(len(heatmap_skills)))
ax.set_yticks(range(len(heatmap_skills)))
ax.set_xticklabels(heatmap_skills, rotation=90)
ax.set_yticklabels(heatmap_skills)
ax.set_title("Co-occurrence counts for the 20 most prevalent skills")
fig.colorbar(im, ax=ax, label="Number of advertisements listing both skills")
plt.tight_layout()
plt.show()

In [ ]:
skill_counts.to_csv("RQ3_skill_prevalence.csv", index=False)
category_summary.to_csv("RQ3_category_prevalence.csv", index=False)
pairs.to_csv("RQ3_skill_cooccurrence.csv", index=False)

print("Saved:")
print("- RQ3_skill_prevalence.csv")
print("- RQ3_category_prevalence.csv")
print("- RQ3_skill_cooccurrence.csv")

from google.colab import files
files.download("RQ3_skill_prevalence.csv")
files.download("RQ3_category_prevalence.csv")
files.download("RQ3_skill_cooccurrence.csv")